# PAP Emulation - MuJoCo MJX GPU/TPU Lab

Copyright © 1993-2026 Abhishek Choudhary. All rights reserved. AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary. Licensed under GNU GPL v3.0. Third-party dependencies retain their own licenses.

Vectorized articulated-body dynamics using the official MuJoCo MJX/JAX API. Runs on supported CPU/GPU/TPU hardware and records the actual JAX backend.

**Author:** Abhishek Choudhary · **Affiliation:** AyeAI · **ORCID:** [0009-0002-0684-8320](https://orcid.org/0009-0002-0684-8320)

**Deployment:** browser-only platform; this notebook runs in a user-controlled Colab runtime. GPU/TPU availability is variable and not guaranteed. **Evidence boundary:** results remain `SIMULATED` and `UNVERIFIED`; this is not HIL or physical verification.

## 0. Runtime, security and provenance
Run cells in order. The notebook records backend versions, solver settings, runtime device, model configuration, timestamps and SHA-256 job/result digests. Results must be imported through the separate PAP Colab Bridge. Never upload untrusted native FMUs or binaries.

In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary. All rights reserved.
# AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.
# SPDX-License-Identifier: GPL-3.0-only
# Embedded in each notebook so notebooks remain independently runnable.
PAP_BRIDGE_VERSION = '1.0.0'
PAP_JOB_SCHEMA = 'pap.emulation.job/1.0'
PAP_RESULT_SCHEMA = 'pap.emulation.result/1.0'

def pap_canonical(value):
    import json
    return json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False, allow_nan=False)

def pap_digest(text):
    import hashlib
    return 'sha256:' + hashlib.sha256(text.encode('utf-8')).hexdigest()

def pap_make_job(backend_id, operation, inputs):
    from datetime import datetime, timezone
    body = {
        'schema': PAP_JOB_SCHEMA,
        'bridgeVersion': PAP_BRIDGE_VERSION,
        'createdAt': datetime.now(timezone.utc).isoformat(),
        'target': {'backendId': backend_id, 'backendVersion': None, 'operation': operation, 'requestedDevice': 'auto'},
        'sir': {'header': {'id': 'colab-reference-fixture', 'irVersion': '1.0.0'}, 'components': []},
        'inputs': inputs,
        'observations': None,
        'configuration': {},
        'provenance': {'source': 'COLAB_REFERENCE_FIXTURE', 'sourceVersion': '1.0.0', 'license': 'GPL-3.0-only'}
    }
    canonical = pap_canonical(body)
    digest = pap_digest(canonical)
    return {**body, 'jobId': 'papjob-' + digest[7:27], 'digest': digest, 'canonicalPayload': canonical}

def pap_verify_job(job, expected_backend):
    import json
    if not isinstance(job, dict) or job.get('schema') != PAP_JOB_SCHEMA:
        raise ValueError('Unsupported/missing PAP job schema')
    payload = job.get('canonicalPayload')
    if not isinstance(payload, str): raise ValueError('Job canonicalPayload missing')
    parsed = json.loads(payload)
    body = {k:v for k,v in job.items() if k not in ('jobId','digest','canonicalPayload')}
    if pap_canonical(parsed) != pap_canonical(body): raise ValueError('Job payload/body mismatch')
    digest = pap_digest(payload)
    if digest != job.get('digest') or job.get('jobId') != 'papjob-' + digest[7:27]:
        raise ValueError('Job digest/jobId verification failed')
    if job.get('target',{}).get('backendId') != expected_backend:
        raise ValueError('Job targets a different backend')
    if not isinstance(job.get('inputs'), dict): raise ValueError('Job inputs must be an object')
    return job

def pap_get_job(backend_id, operation, default_inputs, use_demo=True):
    if use_demo:
        return pap_make_job(backend_id, operation, default_inputs), True
    from google.colab import files
    import json
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.lower().endswith('.json')]
    if len(candidates) != 1: raise ValueError('Upload exactly one browser-exported PAP job .json')
    return pap_verify_job(json.loads(uploaded[candidates[0]].decode('utf-8')), backend_id), False

def pap_make_result(job, backend_id, backend_version, outputs, diagnostics, device='cpu'):
    from datetime import datetime, timezone
    body = {
        'schema': PAP_RESULT_SCHEMA,
        'bridgeVersion': PAP_BRIDGE_VERSION,
        'jobId': job['jobId'],
        'jobDigest': job['digest'],
        'backend': {'id': backend_id, 'version': str(backend_version)},
        'execution': {'status': 'EXECUTED', 'runtime': 'COLAB', 'startedAt': None,
                      'finishedAt': datetime.now(timezone.utc).isoformat(), 'device': str(device)},
        'outputs': outputs,
        'diagnostics': diagnostics,
        'fidelity': 'SIMULATED',
        'physicalVerification': False,
        'evidenceState': 'UNVERIFIED',
        'claims': {'verified': False, 'hil': False, 'physical': False}
    }
    canonical = pap_canonical(body)
    return {**body, 'canonicalPayload': canonical, 'resultDigest': pap_digest(canonical)}

def pap_export(job, result, is_demo, result_filename='pap-emulation-result.json'):
    import json
    with open(result_filename, 'w', encoding='utf-8') as stream:
        json.dump(result, stream, indent=2, ensure_ascii=False, allow_nan=False)
    if is_demo:
        with open('pap-emulation-job.json', 'w', encoding='utf-8') as stream:
            json.dump(job, stream, indent=2, ensure_ascii=False, allow_nan=False)
    try:
        from google.colab import files
        files.download(result_filename)
        if is_demo: files.download('pap-emulation-job.json')
    except ImportError:
        print('Saved locally:', result_filename, '(download manually outside Colab)')
    print('Result remains SIMULATED / UNVERIFIED. Import it through the PAP Colab Bridge.')


In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary. All rights reserved. AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.
# SPDX-License-Identifier: GPL-3.0-only
%pip -q install 'mujoco>=3.2' 'mujoco-mjx>=3.2'

In [ ]:
import sys, platform, time, jax, jax.numpy as jnp, mujoco
from mujoco import mjx
print({'python':sys.version,'platform':platform.platform(),'mujoco':mujoco.__version__,'jax':jax.__version__,'devices':[str(d) for d in jax.devices()]})
PAP_USE_DEMO_JOB=True
default_inputs={'environmentCount':256,'steps':250,'dt':0.002,'angleMin':-0.5,'angleMax':0.5}
job,job_is_demo=pap_get_job('mujoco.mjx','mujoco.mjx.batch_pendulum',default_inputs,PAP_USE_DEMO_JOB)

## 1. MJX-JAX vectorized pendulum
MJX-JAX supports JAX/XLA compute devices, including supported NVIDIA/AMD GPUs and Google Cloud TPUs. MJX-Warp is an alternative NVIDIA GPU implementation with different feature and autodiff trade-offs. This notebook uses the documented MJX-JAX `put_model`, `make_data`, `step`, `vmap` and `jit` path.

In [ ]:
xml='''<mujoco model="pap-pendulum"><option timestep="0.002" gravity="0 0 -9.81"/><worldbody><body name="bob"><joint name="hinge" type="hinge" axis="0 1 0"/><geom type="capsule" fromto="0 0 0 0 0 -1" size="0.04" mass="1"/></body></worldbody></mujoco>'''
mjm=mujoco.MjModel.from_xml_string(xml); model=mjx.put_model(mjm)
N=int(job['inputs'].get('environmentCount',256)); steps=int(job['inputs'].get('steps',250)); dt=float(job['inputs'].get('dt',mjm.opt.timestep))
if N<1 or N>100000 or steps<1 or steps>100000 or abs(dt-mjm.opt.timestep)>1e-12: raise ValueError('Invalid environment count/step count/timestep')
angles=jnp.linspace(float(job['inputs'].get('angleMin',-0.5)),float(job['inputs'].get('angleMax',0.5)),N,dtype=jnp.float32)
@jax.vmap
def simulate_one(angle):
 data=mjx.make_data(model)
 data=data.replace(qpos=data.qpos.at[0].set(angle))
 data=jax.lax.fori_loop(0,steps,lambda i,d:mjx.step(model,d),data)
 return data.qpos[0],data.qvel[0]
compiled=jax.jit(simulate_one); started=time.time(); qpos,qvel=compiled(angles); jax.block_until_ready(qpos); elapsed=time.time()-started
assert bool(jnp.isfinite(qpos).all()) and bool(jnp.isfinite(qvel).all())
print({'environments':N,'steps':steps,'elapsedSeconds':elapsed,'jaxBackend':jax.default_backend(),'finalQposRange':[float(jnp.min(qpos)),float(jnp.max(qpos))]})

## 2. Result export and validation boundary
This is a numerical execution smoke case, not a calibrated mechanism validation. Production validation requires energy drift, timestep refinement, constraint/contact benchmarks and documented MJX feature parity for the model being used.

In [ ]:
outputs=[{'sequence':0,'time':float(steps*dt),'values':{'qposMin':float(jnp.min(qpos)),'qposMax':float(jnp.max(qpos)),'qvelMin':float(jnp.min(qvel)),'qvelMax':float(jnp.max(qvel))}}]
result=pap_make_result(job,'mujoco.mjx',mujoco.__version__,outputs,{'model':'single-hinge-pendulum','environmentCount':N,'steps':steps,'dt':dt,'elapsedSeconds':elapsed,'jaxVersion':jax.__version__,'jaxBackend':jax.default_backend(),'devices':[str(d) for d in jax.devices()],'gpuOrTpuUsed':jax.default_backend()!='cpu'},jax.default_backend())
pap_export(job,result,job_is_demo,'pap-mjx-result.json')